# Resume Evaluation AI — Part 2: RAG Feedback Generation & Conversational Interface
**Name:** Abhiranjan Sharma

Builds on Part 1 (01_data_prep_and_matching.ipynb). This notebook covers:
1. A curated knowledge base of skill/role advice
2. A FAISS retrieval index over that knowledge base
3. Retrieval-grounded, explainable feedback generation
4. An Ollama-backed LLM generation layer (+ fully offline template fallback)
5. The conversational interface (app.py, run separately as a Gradio app)

**A note on the LLM layer:** this notebook was built in a sandboxed
environment that cannot reach localhost:11434 (your local Ollama
server) or huggingface.co(for downloading sentence-transformer
weights). Both integrations are written and tested against their real
APIs / fallback logic, but the *live* LLM calls will only activate when
you run this on your own machine with Ollama running. Everything falls
back gracefully to offline behavior otherwise, so every cell below
still produces real, useful output.

In [1]:
import sys, json
sys.path.insert(0, '.')

from src.data_prep import load_skill_bank, load_and_clean
from src.embeddings import EmbeddingBackend
from src.matcher import ResumeMatcher

DATA_PATH = "resumes_dataset.jsonl"

skill_bank = load_skill_bank()
df = load_and_clean(DATA_PATH)

backend = EmbeddingBackend(prefer_transformer=True)
matcher = ResumeMatcher(skill_bank, embedding_backend=backend)
matcher.fit(df['clean_text'].tolist())
print(f"Matcher ready. Embedding backend: {backend.mode}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Matcher ready. Embedding backend: sentence-transformer


## 1. Knowledge Base

In [2]:
from src.rag_knowledge import build_knowledge_base, RAGIndex

kb_docs = build_knowledge_base(skill_bank)
print(f"Knowledge base: {len(kb_docs)} documents")
print()
for doc in kb_docs[:3]:
    print(f"[{doc.doc_id}] {doc.text[:120]}")


Knowledge base: 213 documents

[role::Java Developer] For a Java Developer role, employers typically look for a combination of the core required skills, demonstrated through 
[skill::Java Developer::java] [Java Developer] java: Java is a mainstay of enterprise backend systems, prized for its portability and mature tooling. 
[skill::Java Developer::spring] [Java Developer] spring: Spring (and Spring Boot) is the standard framework for enterprise Java applications, handling d


## 2. FAISS Retrieval Index

In [3]:
rag_backend = EmbeddingBackend(prefer_transformer=True)
rag_backend.fit([d.text for d in kb_docs])
rag = RAGIndex(rag_backend).build(kb_docs)

# quick retrieval sanity check
results = rag.retrieve("missing python and machine learning skills for a data science role", top_k=4)
for doc, score in results:
    print(f"[{score:.3f}] {doc.doc_id}")
    print(f"    {doc.text[:140]}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[0.680] role::Data Science
    For a Data Science role, employers typically look for a combination of the core required skills, demonstrated through real projects or work 
[0.620] role::Python Developer
    For a Python Developer role, employers typically look for a combination of the core required skills, demonstrated through real projects or w
[0.608] skill::Machine Learning Engineer::python
    [Machine Learning Engineer] python: Python is a widely used general-purpose language valued for readability and its large ecosystem (data, w
[0.563] skill::Data Science::machine learning
    [Data Science] machine learning: Machine learning skills are demonstrated through applied projects: data cleaning, model selection, evaluati


## 3. Ollama LLM Client

checks the local server before attempting
generation, so the pipeline degrades when Ollama isn't
running.

In [4]:
from src.feedback_generator import OllamaClient, generate_feedback

ollama = OllamaClient()
print(f"Ollama reachable at {ollama.host}: {ollama.is_available()}")
print(f"Configured model: {ollama.model}")
print()
print("(If this says False, either Ollama isn't running, or you're executing")
print(" this notebook in an environment without access to your local machine.")
print(" Start Ollama locally with `ollama serve` and ensure the model is pulled")
print(" with `ollama pull gemma4:e4b` to activate live LLM generation.)")


Ollama reachable at http://localhost:11434: True
Configured model: gemma4:e4b

(If this says False, either Ollama isn't running, or you're executing
 this notebook in an environment without access to your local machine.
 Start Ollama locally with `ollama serve` and ensure the model is pulled
 with `ollama pull gemma4:e4b` to activate live LLM generation.)


## 4. Explainable Feedback Generation


In [5]:
sample = df[df['Category']=='Java Developer'].iloc[3]
match = matcher.match(sample['Text'], 'Java Developer')
result = generate_feedback(match, rag, ollama_client=ollama)

print(f"[feedback source: {result['source']}]\n")
print(result['feedback'])


[feedback source: ollama]

This is a very strong profile, and the results reflect that! Achieving 100.0% skill coverage and having no missing skills for the Java Developer role is highly impressive. You have demonstrated proficiency across critical technologies, including core programming languages like **java** and **javascript**, foundational frameworks such as **spring** and **jquery**, and critical database management using both **oracle** and **mysql**.

Your technical breadth is excellent, and your additional skills—like **architecture**, **data analysis**, and **jenkins**—show a holistic understanding of the development lifecycle that goes beyond the core requirements.

Since your skill matching is complete, the feedback shifts from *what* you know to *how* you prove it. The knowledge base emphasizes that employers want evidence of applying these skills in a team or production context. To maximize your chances, focus your resume and interview narratives on demonstrating the *imp

In [6]:
# A mismatched example to see how feedback adapts to a larger skill gap
sample2 = df[df['Category']=='Testing'].iloc[2]
match2 = matcher.match(sample2['Text'], 'Machine Learning Engineer')
result2 = generate_feedback(match2, rag, ollama_client=ollama)

print(f"[feedback source: {result2['source']}]\n")
print(result2['feedback'])


[feedback source: ollama]

Overall, this review indicates a significant knowledge gap for the Machine Learning Engineer role, as reflected by the 0.0/100 skill coverage. However, please view this feedback not as a final judgment, but as a clear roadmap for your professional development. You already possess experience with *monitoring*, which is a valuable operational skill; the next step is demonstrating how that monitoring integrates into a full ML production lifecycle.

To strengthen your profile, you need to build out demonstrable skills in the core ML pipeline. Your resume should focus on projects that apply foundational tools like **python** and operationalize models using frameworks such as **tensorflow** or **pytorch**. Crucially, the role requires more than theoretical knowledge—you must show practical application of DevOps principles.

Specifically, focus on building projects that integrate **docker** and containerization, deploying them to cloud platforms like **aws** or **gc

## 5. Sample Feedback Across Multiple Resumes

A quick qualitative pass across a handful of resumes/roles to sanity check
that scores and feedback text move sensibly together (higher score →
more matched skills referenced positively; lower score → gap-focused
feedback).

In [7]:
import random
random.seed(7)
sample_rows = df.sample(5, random_state=7)

for _, row in sample_rows.iterrows():
    m = matcher.match(row['Text'], row['Category'])  # match against own true category
    r = generate_feedback(m, rag, ollama_client=ollama)
    print(f"=== {row['Category']} | score {m.overall_score}/100 | source: {r['source']} ===")
    print(r['feedback'][:300].replace(chr(10), ' '))
    print()


=== DevOps | score 68.5/100 | source: ollama ===
This is a strong foundation for a DevOps role! Your existing skill set—particularly your experience with **AWS, Docker, Jenkins, Python, Ansible, Git,** and **CI/CD**—demonstrates significant technical capability. Furthermore, the additional skills you possess, such as **Grafana, IAM,** and **Archit

=== React Developer | score 46.7/100 | source: ollama ===
Overall, you have a strong technical foundation that positions you well for a developer role, demonstrated by your proficiency in technologies like **javascript**, **html**, **css**, and version control using **git**. Furthermore, your experience with **agile** methodologies and tools like **jenkins

=== Software Developer | score 64.2/100 | source: ollama ===
This is a very strong foundation for a Software Developer role, and your proficiency across full-stack technologies is evident. You have excellent command of key enterprise tools, including `git`, cloud platforms like `aws`, and

## 5b. Reverse Search: Find Candidates for a Job Description

In [8]:
matcher.fit_candidate_pool(df, metadata_cols=['Category', 'Source'])
print(f"Candidate pool indexed: {len(df)} resumes.")

Candidate pool indexed: 3500 resumes.


In [9]:
custom_jd = '''
We are hiring a Python Developer with strong experience in Django,
building REST APIs, SQL databases, and deploying to AWS using Docker.
Experience with CI/CD pipelines and Git-based workflows is a plus.
'''

jd_required, ranked = matcher.rank_candidates_for_jd(custom_jd, top_k=5)
print(f"Detected required skills from JD: {jd_required}\n")

for rank, r in enumerate(ranked, start=1):
    print(f"#{rank} | score {r.overall_score}/100 | {r.metadata} | "
          f"matched: {r.matched_skills} | missing: {r.missing_skills}")

Detected required skills from JD: ['aws', 'ci/cd', 'django', 'docker', 'git', 'hiring', 'python', 'sql']

#1 | score 70.2/100 | {'Category': 'Backend Developer', 'Source': 'Synthetic'} | matched: ['aws', 'ci/cd', 'django', 'docker', 'git', 'python'] | missing: ['hiring', 'sql']
#2 | score 70.0/100 | {'Category': 'Backend Developer', 'Source': 'Synthetic'} | matched: ['aws', 'ci/cd', 'django', 'docker', 'python', 'sql'] | missing: ['git', 'hiring']
#3 | score 68.9/100 | {'Category': 'Backend Developer', 'Source': 'Synthetic'} | matched: ['aws', 'ci/cd', 'django', 'docker', 'python', 'sql'] | missing: ['git', 'hiring']
#4 | score 68.9/100 | {'Category': 'Backend Developer', 'Source': 'Synthetic'} | matched: ['aws', 'ci/cd', 'django', 'docker', 'python', 'sql'] | missing: ['git', 'hiring']
#5 | score 68.6/100 | {'Category': 'Backend Developer', 'Source': 'Synthetic'} | matched: ['aws', 'ci/cd', 'django', 'docker', 'python', 'sql'] | missing: ['git', 'hiring']


In [10]:
# A second example with a very different JD, to confirm ranking adapts
devops_jd = '''
DevOps engineer needed with hands-on Kubernetes, Docker, Terraform,
Jenkins, and AWS experience. Strong Linux administration skills and
CI/CD pipeline ownership required.
'''

jd_required2, ranked2 = matcher.rank_candidates_for_jd(devops_jd, top_k=5)
print(f"Detected required skills from JD: {jd_required2}\n")
for rank, r in enumerate(ranked2, start=1):
    print(f"#{rank} | score {r.overall_score}/100 | {r.metadata} | matched: {r.matched_skills}")

Detected required skills from JD: ['aws', 'ci/cd', 'devops', 'docker', 'jenkins', 'kubernetes', 'linux', 'terraform']

#1 | score 77.3/100 | {'Category': 'DevOps', 'Source': 'ResumeAtlas'} | matched: ['aws', 'devops', 'docker', 'jenkins', 'kubernetes', 'linux', 'terraform']
#2 | score 77.3/100 | {'Category': 'DevOps', 'Source': 'ResumeAtlas'} | matched: ['aws', 'devops', 'docker', 'jenkins', 'kubernetes', 'linux', 'terraform']
#3 | score 76.6/100 | {'Category': 'DevOps', 'Source': 'ResumeAtlas'} | matched: ['aws', 'devops', 'docker', 'jenkins', 'kubernetes', 'linux', 'terraform']
#4 | score 76.4/100 | {'Category': 'DevOps', 'Source': 'ResumeAtlas'} | matched: ['aws', 'devops', 'docker', 'jenkins', 'kubernetes', 'linux', 'terraform']
#5 | score 76.3/100 | {'Category': 'DevOps', 'Source': 'ResumeAtlas'} | matched: ['aws', 'devops', 'docker', 'jenkins', 'kubernetes', 'linux', 'terraform']


Both searches correctly surface candidates from the matching category
(Python/Backend-leaning roles for the first JD, DevOps for the second),
confirming the reverse-search direction behaves consistently with the
resume→role matching validated in Part 1.

## 6. Conversational Interface

In [11]:
# Preview app.py's structure (already tested standalone; not launched here since
# Gradio's blocking server isn't suitable to run inside a notebook cell)
with open('app.py') as f:
    print(f.read()[:1200])
print('... (see app.py for full source)')


"""
Resume Evaluation AI -- Conversational Interface (Gradio)

Run locally with:
    python app.py

Requires Ollama running locally (http://localhost:11434) with the model
set in src/feedback_generator.py (default: gemma4:e4b) for LLM-generated
chat responses. Falls back to template-based responses if Ollama isn't
reachable, so the app still works out of the box.
"""
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import gradio as gr
from src.data_prep import load_skill_bank, load_and_clean
from src.embeddings import EmbeddingBackend
from src.matcher import ResumeMatcher
from src.rag_knowledge import build_knowledge_base, RAGIndex
from src.feedback_generator import OllamaClient, generate_feedback, FEEDBACK_SYSTEM_PROMPT
from src.pdf_reader import extract_resume_text

DATA_PATH = os.environ.get("RESUME_DATA_PATH", "resumes_dataset.jsonl")

print("Loading pipeline (this fits the model once at startup)...")
skill_bank = load_skill_bank()
df = load_and_

## 7. Summary

End-to-end pipeline: resume text → skill extraction → embedding + skill-bank
match score → RAG-grounded explainable feedback → conversational
follow-up, all with offline-safe fallbacks at every stage requiring
external services (embeddings, LLM generation).